In [ ]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import StructType, StructField, LongType, StringType, TimestampType ,IntegerType
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, get_json_object, current_timestamp,lit


In [ ]:
spark = SparkSession.builder.getOrCreate()

In [ ]:
schema = StructType([
    StructField("timestamp", LongType(), True),
    StructField("itemid", IntegerType(), True),
    StructField("property", IntegerType(), True),
    StructField("value", StringType(), True)
])


In [ ]:


bronze_path = "s3://databricks-batch-data-demo/raw/"
checkpoint_base = "/Volumes/dev/bronze/checkpoint/_checkpoints/bronze_ingestion/"

In [ ]:
df = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation", checkpoint_base+"schema") \
    .option("header", "true") \
    .schema(schema) \
    .load(bronze_path)




In [ ]:

df = df.select(
    "*", 
    current_timestamp().alias("_ingestion_time"),
    lit("manual_csv").alias("_source_file")
)


In [ ]:

query = (df.writeStream
         .format("delta")
         .outputMode("append")
         .option("checkpointLocation", checkpoint_base + "data")
         .option("mergeSchema", "true")
         .trigger(availableNow=True)
         .toTable("dev.bronze.batch_bronze"))


In [ ]:
%sql
select * from dev.bronze.batch_bronze 
limit 1

